In [1]:
import os
os.environ['picaso_refdata'] = '/Users/maria/picaso_reference'
os.environ['PYSYN_CDBS'] = '/Users/maria/picaso_reference/stellar_grids'

from picaso import justdoit as jdi
from picaso import justplotit as jpi
import numpy as np

jpi.output_notebook()

Loading BokehJS ...

In [3]:
opa = jdi.opannection(wave_range=[0.6, 5.5])

planet = jdi.inputs()
planet.gravity(mass=0.28, mass_unit=jdi.u.Unit('M_jup'),
               radius=1.27, radius_unit=jdi.u.Unit('R_jup'))

planet.star(opa, temp=5400, metal=0.0, logg=4.5,
            radius=0.895, radius_unit=jdi.u.Unit('R_sun'),
            database='phoenix')

planet.atmosphere(filename=jdi.HJ_pt(), sep=r'\s+')

In [11]:
cloud_fractions = [0.0, 0.25, 0.50, 0.75, 1.0]

wnos = []
spectra = []

# First compute the fully clear spectrum
p_clear = jdi.inputs()
p_clear.gravity(mass=0.28, mass_unit=jdi.u.Unit('M_jup'),
                radius=1.27, radius_unit=jdi.u.Unit('R_jup'))
p_clear.star(opa, temp=5400, metal=0.0, logg=4.5,
             radius=0.895, radius_unit=jdi.u.Unit('R_sun'),
             database='phoenix')
p_clear.atmosphere(filename=jdi.HJ_pt(), sep=r'\s+')
p_clear.clouds(p=[1], dp=[4], opd=[0], g0=[0], w0=[0])
df_clear = p_clear.spectrum(opa, full_output=True, calculation='transmission')
wno, rprs2_clear = df_clear['wavenumber'], df_clear['transit_depth']
wno, rprs2_clear = jdi.mean_regrid(wno, rprs2_clear, R=150)

# Then compute the fully cloudy spectrum
p_cloudy = jdi.inputs()
p_cloudy.gravity(mass=0.28, mass_unit=jdi.u.Unit('M_jup'),
                 radius=1.27, radius_unit=jdi.u.Unit('R_jup'))
p_cloudy.star(opa, temp=5400, metal=0.0, logg=4.5,
              radius=0.895, radius_unit=jdi.u.Unit('R_sun'),
              database='phoenix')
p_cloudy.atmosphere(filename=jdi.HJ_pt(), sep=r'\s+')
p_cloudy.clouds(p=[1], dp=[4], opd=[10], g0=[0], w0=[0])
df_cloudy = p_cloudy.spectrum(opa, full_output=True, calculation='transmission')
wno_c, rprs2_cloudy = df_cloudy['wavenumber'], df_cloudy['transit_depth']
wno_c, rprs2_cloudy = jdi.mean_regrid(wno_c, rprs2_cloudy, R=150)

# Mix them by cloud fraction
for cf in cloud_fractions:
    mixed = (1 - cf) * rprs2_clear + cf * rprs2_cloudy
    wnos.append(wno)
    spectra.append(mixed)

print("Done!")

Done!


In [12]:
import pandas as pd

labels = ['0% clouds (clear)', '25% clouds', '50% clouds', '75% clouds', '100% clouds']

jpi.show(jpi.spectrum(wnos, spectra, legend=labels, plot_width=600,
                      x_axis_label='Wavelength [μm]',
                      y_axis_label='Transit Depth (Rp/Rs)²'))